# Imports

In [ ]:
import io
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

# DATA INLADEN 

In [ ]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("/Users/nachatissa/Desktop/SCHOOL/SEM2/Advanced AI/untitled folder/Advanced-AI/dataset/closed-open-eyes/data")

parquet_files = sorted(DATA_DIR.rglob("*.parquet"))
df = pd.concat([pd.read_parquet(p) for p in parquet_files], ignore_index=True)

print(df.shape)
print(df.head())

# CACHING IMAGE RESIZE to make model training faster RUN ONLY ONCE 

In [ ]:
from pathlib import Path
from PIL import Image
import io

CACHE_DIR = Path("/Users/nachatissa/Desktop/SCHOOL/SEM2/Advanced AI/untitled folder/Advanced-AI/dataset/eye_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

rows = []

for i, row in df.iterrows():
    img = Image.open(io.BytesIO(row["Image_data"]["file"])).convert("RGB")
    img = img.resize((224, 224))
    out_path = CACHE_DIR / f"{i}.jpg"
    img.save(out_path, quality=95)
    rows.append({"path": str(out_path), "Label": row["Label"]})

In [ ]:
cached_df = pd.DataFrame(rows)
cached_df.to_csv("/Users/nachatissa/Desktop/SCHOOL/SEM2/Advanced AI/untitled folder/Advanced-AI/dataset/eye_cache.csv", index=False)

print(cached_df.shape)
print(cached_df.head())

# Label map + test image cell


In [ ]:
label_map = {"closed_eyes": 0, "open_eyes": 1}

sample = df.iloc[0]
img = Image.open(io.BytesIO(sample["Image_data"]["file"])).convert("RGB")
print(img.size)
plt.imshow(img)
plt.title(sample["Label"])
plt.axis("off")
plt.show()

# Simple dataset


In [ ]:
class EyeDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["path"]).convert("RGB")
        img = np.array(img, dtype=np.float32) / 255.0
        img = torch.tensor(img).permute(2, 0, 1)
        label = torch.tensor(label_map[row["Label"]], dtype=torch.long)
        return img, label

# Split and loaders

In [ ]:
print(cached_df.columns)
print(cached_df.head())

train_df, val_df = train_test_split(
    cached_df,
    test_size=0.2,
    random_state=42,
    stratify=cached_df["Label"]
)

train_ds = EyeDataset(train_df)
val_ds = EyeDataset(val_df)

In [ ]:
train_ds = EyeDataset(train_df)
val_ds = EyeDataset(val_df)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32)

# CNN CLASS

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Flatten(),
            nn.Linear(64 * 28 * 28, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        return self.model(x)

# device training setup 

In [ ]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using:", device)

model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# TRAIN 

In [ ]:
for epoch in range(5):
    model.train()
    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        preds = outputs.argmax(1)
        total += labels.size(0)
        correct += (preds == labels).sum().item()

    print(f"Epoch {epoch+1} | loss={running_loss:.4f} | acc={correct/total:.4f}")

# VALIDATE

In [ ]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        preds = outputs.argmax(1)
        total += labels.size(0)
        correct += (preds == labels).sum().item()

print(f"Validation accuracy: {correct/total:.4f}")

TESTING ON NEW DATA

In [108]:
# 1-SECOND FATIGUE ALARM w/ MULTIPLE SOUND OPTIONS
import cv2
import urllib.request
import os
import numpy as np
from PIL import Image
import torch
import subprocess
import os

# Download cascade
cascade_url = "https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_frontalface_default.xml"
cascade_path = "haarcascade_frontalface_default.xml"
if not os.path.exists(cascade_path):
    urllib.request.urlretrieve(cascade_url, cascade_path)

face_cascade = cv2.CascadeClassifier(cascade_path)

# Fatigue (1 second)
closed_frames = 0
ALARM_FRAMES = 30  # 1 second!
last_alarm = 0

cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 900)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 700)

model.eval()
print("🚨 1s Alarm | Pick sound method below:")

while True:
    ret, frame = cap.read()
    if not ret: break
    
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.1, 4)
    
    eyes_closed = True
    
    for (x, y, w, h) in faces:
        face_crop = frame[y:y+h, x:x+w]
        if face_crop.size == 0: continue
        
        face_rgb = cv2.cvtColor(face_crop, cv2.COLOR_BGR2RGB)
        img = Image.fromarray(face_rgb).resize((224, 224))
        img_array = np.array(img, dtype=np.float32) / 255.0
        img_tensor = torch.tensor(img_array).permute(2, 0, 1).unsqueeze(0).to(device)
        
        pred = model(img_tensor).argmax(1).item()
        confidence = torch.softmax(model(img_tensor), 1).max().item()
        
        if pred == 1:  # OPEN
            eyes_closed = False
            closed_frames = 0
        else:  # CLOSED
            closed_frames += 1
        
        label = f"CLOSED {confidence:.0%}" if pred == 0 else f"OPEN {confidence:.0%}"
        color = (0, 0, 255) if pred == 0 else (0, 255, 0)
        cv2.rectangle(frame, (x, y), (x+w, y+h), color, 4)
        cv2.putText(frame, label, (x, y-15), cv2.FONT_HERSHEY_SIMPLEX, 1, color, 3)
    
    # Timer
    fatigue_time = closed_frames / 30
    cv2.putText(frame, f"Tired: {fatigue_time:.1f}s", (30, 70), 
                cv2.FONT_HERSHEY_SIMPLEX, 1.3, (0, 255, 255), 3)
    
    # 🚨 1s ALARM!
    if closed_frames >= ALARM_FRAMES:
        cv2.putText(frame, "🚨🚨🚨 WAKE UP NOW!! 🚨🚨🚨", 
                    (30, 160), cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 0, 255), 5)
        
        if closed_frames - last_alarm > 30:  # Every 1s
            last_alarm = closed_frames
            
            # 🔥 METHOD 1: Mac say (Terminal tab)
            print("🔊 ALARM!")
            os.system("say -v Samantha 'WAKE UP!'")
            
            # 🔥 METHOD 2: Terminal beep (works everywhere)
            print("\a")  # ASCII bell
            
            # 🔥 METHOD 3: Run in terminal (copy this):
            # os.system("afplay /System/Library/Sounds/Ping.aiff")
    
    cv2.imshow('🚨 1s Fatigue Alarm', frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

🚨 1s Alarm | Pick sound method below:
🔊 ALARM!

